# 01 - Preprocessing
Prepares the core dataset into model-ready variants.
Each model has different data format requirements -- all handled here.

| Model | Key Requirement |
|-------|----------------|
| XGBoost | Engineered features, no NaN |
| SARIMAX | Clean target series + exogenous columns |
| Prophet | Columns renamed to `ds` and `y` |
| LSTM/RNN | Scaled, sequenced arrays |

- Version 1.02  
- updated 30.04.26

## Imports

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import sys
sys.path.append(r'Q:\scripts\projects\ts-model-framework')
import config
import warnings
warnings.filterwarnings('ignore')

print(f"Project path: {config.PROJECT_PATH}")
print(f"Data path: {config.DATA_PATH}")
print(f"Data path: {config.EXPORTS_PATH}")
print("Libraries loaded.")

Project path: Q:\scripts\projects\ts-model-framework
Data path: Q:\scripts\projects\ts-model-framework\data
Data path: Q:\scripts\projects\ts-model-framework\exports
Libraries loaded.


---
## S1 - Load Core Dataset
Single source of truth -- all model variants derived from this.

In [9]:
# FILE_NAME = "timeseries_with_features.csv"
# DATE_COLUMN = "date"
# TARGET_COLUMN = "unit_sales"

file_path = os.path.join(config.DATA_PATH, config.FILE_NAME)
df_core = pd.read_csv(file_path, parse_dates=[config.DATE_COLUMN], index_col=config.DATE_COLUMN)
df_core = df_core.sort_index()

# Check for missing dates and fill gaps
full_range   = pd.date_range(start=df_core.index.min(), end=df_core.index.max(), freq='D')
missing_dates = full_range.difference(df_core.index)
print(f"Missing dates found: {len(missing_dates)}")

if len(missing_dates) > 0:
    df_core = df_core.reindex(full_range)
    # df_core = df_core.ffill
    df_core = df_core.fillna(0)
    print(f"Missing dates filled with zeros.")

print(f"Core dataset loaded: {df_core.shape}")
print(f"Date range: {df_core.index.min()} to {df_core.index.max()}")

Missing dates found: 2
Missing dates filled with zeros.
Core dataset loaded: (454, 24)
Date range: 2013-01-02 00:00:00 to 2014-03-31 00:00:00


---
## S2 - Train / Test Split
Shared split used across ALL models for fair comparison.
Adjust TEST_START to match your dataset.

In [10]:
TEST_START = "2014-01-01"   # adjust as needed

df_train = df_core[df_core.index < TEST_START].copy()
df_test  = df_core[df_core.index >= TEST_START].copy()

print(f"Train: {df_train.shape} | {df_train.index.min()} to {df_train.index.max()}")
print(f"Test:  {df_test.shape}  | {df_test.index.min()} to {df_test.index.max()}")

Train: (364, 24) | 2013-01-02 00:00:00 to 2013-12-31 00:00:00
Test:  (90, 24)  | 2014-01-01 00:00:00 to 2014-03-31 00:00:00


---
## S3 - XGBoost Variant
Requires: engineered features, no NaN rows.

In [11]:
FEATURES = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'week_of_year',
    'is_weekend', 'is_month_start', 'is_month_end',
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_7d_mean', 'rolling_14d_mean', 'rolling_30d_mean', 'rolling_7d_std',
    'dcoilwtico', 'oil_lag_1', 'oil_rolling_7d_mean',
    'is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
]

# Drop rows where any feature is NaN
xgb_train = df_train[FEATURES + [config.TARGET_COLUMN]].dropna()
xgb_test  = df_test[FEATURES + [config.TARGET_COLUMN]].dropna()

X_train = xgb_train[FEATURES]
y_train = xgb_train[config.TARGET_COLUMN]
X_test  = xgb_test[FEATURES]
y_test  = xgb_test[config.TARGET_COLUMN]

print(f"XGBoost train: {X_train.shape} | test: {X_test.shape}")

XGBoost train: (364, 23) | test: (90, 23)


---
## S4 - SARIMAX Variant
Requires: clean target series + optional exogenous columns.
NaN values forward-filled.

In [12]:
EXOG_COLUMNS = ['dcoilwtico', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday']

sarimax_train = df_train[[config.TARGET_COLUMN] + EXOG_COLUMNS].fillna(method='ffill')
sarimax_test  = df_test[[config.TARGET_COLUMN] + EXOG_COLUMNS].fillna(method='ffill')

y_sarimax_train = sarimax_train[config.TARGET_COLUMN]
y_sarimax_test  = sarimax_test[config.TARGET_COLUMN]
exog_train      = sarimax_train[EXOG_COLUMNS]
exog_test       = sarimax_test[EXOG_COLUMNS]

print(f"SARIMAX train: {y_sarimax_train.shape} | test: {y_sarimax_test.shape}")
print(f"Exog columns: {EXOG_COLUMNS}")

SARIMAX train: (364,) | test: (90,)
Exog columns: ['dcoilwtico', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday']


---
## S5 - Prophet Variant
Requires: DataFrame with exactly two columns named `ds` (date) and `y` (target).

In [13]:
prophet_train = df_train[[config.TARGET_COLUMN]].reset_index().rename(
    columns={'index': 'ds',config.TARGET_COLUMN: 'y'}
)
prophet_test = df_test[[config.TARGET_COLUMN]].reset_index().rename(
    columns={'index': 'ds', config.TARGET_COLUMN: 'y'}
)

print(f"Prophet train: {prophet_train.shape} | test: {prophet_test.shape}")
print(prophet_train.head(3))

Prophet train: (364, 2) | test: (90, 2)
          ds      y
0 2013-01-02  582.0
1 2013-01-03  310.0
2 2013-01-04  338.0


---
## S6 - LSTM/RNN Variant
Requires: scaled values, reshaped into sequences.
SEQUENCE_LENGTH = how many past timesteps the model sees at once.

In [17]:
SEQUENCE_LENGTH = 30   # adjust as needed

scaler = MinMaxScaler()

# Scale on train only -- apply same scaler to test (prevent data leakage)
train_scaled = scaler.fit_transform(df_train[[config.TARGET_COLUMN]].fillna(method='ffill'))
test_scaled  = scaler.transform(df_test[[config.TARGET_COLUMN]].fillna(method='ffill'))

def make_sequences(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_lstm_train, y_lstm_train = make_sequences(train_scaled, SEQUENCE_LENGTH)
X_lstm_test,  y_lstm_test  = make_sequences(test_scaled,  SEQUENCE_LENGTH)

# Reshape for Keras: (samples, timesteps, features)
X_lstm_train = X_lstm_train.reshape(X_lstm_train.shape[0], X_lstm_train.shape[1], 1)
X_lstm_test  = X_lstm_test.reshape(X_lstm_test.shape[0],  X_lstm_test.shape[1],  1)

print(f"LSTM train shape: {X_lstm_train.shape} | test shape: {X_lstm_test.shape}")
print(f"Scaler range: {scaler.data_min_[0]:.2f} to {scaler.data_max_[0]:.2f}")

LSTM train shape: (334, 30, 1) | test shape: (60, 30, 1)
Scaler range: 0.00 to 1090.00


---
## S7 - Notes & Dataset Observations
Document any preprocessing decisions made for this specific dataset.

- Fill method chosen:
- Exog columns selected and why:
- SEQUENCE_LENGTH rationale:
- Features included/excluded and why: